# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using Croissant
dataset = mlc.Dataset(croissant_url)

# Show metadata information
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}\n")
print(f"Authors: {', '.join([str(a) for a in getattr(metadata, 'author', [])]) if hasattr(metadata, 'author') else ''}")
print(f"Cite As: {getattr(metadata, 'citeAs', '') if hasattr(metadata, 'citeAs') else ''}\n")
print("Keywords:", getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available record sets and their `@id`s, then examine fields and columns for each record set.

In [ ]:
# List available record sets and their field/column @ids
record_sets = dataset.metadata.record_sets
print("Available record sets and their @ids:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}\n  @id: {rs.id}")

    # List fields for this record set
    print("  Fields (columns):")
    for f in getattr(rs, 'fields', []):
        print(f"    - {f.name}: @id={f.id}; dataType={getattr(f, 'data_type', 'unknown')}")
    print()

In [ ]:
# Optionally, preview a few records from the main record set
# Select the main record set by @id (replace this with the actual @id printed above)
# For example: main_recordset_id = 'cr:RecordSet/Proteins' (depending on dataset design)

# Let's get the first record set @id
main_recordset_id = record_sets[0].id if record_sets else None
if main_recordset_id:
    print(f"\nSample records from record set {main_recordset_id}:")
    for i, rec in enumerate(dataset.records(record_set=main_recordset_id)):
        pprint.pprint(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Use the record set and field `@id`s from the previous overview.

In [ ]:
# Extract all main DataFrames
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records for record set '{rsid}'. Columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set '{rsid}'.")

# Preview the first table
if record_set_ids and record_set_ids[0] in dataframes:
    df = dataframes[record_set_ids[0]]
    print(f"Columns available in main DataFrame ({record_set_ids[0]}):\n", df.columns.tolist())
    df.head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. For illustration, select a numeric field and perform cleaning and normalization using the field `@id`.

In [ ]:
# For EDA, select the first record set and its numeric field
import numpy as np

# Get first record set and DataFrame
if record_sets and record_set_ids[0] in dataframes:
    main_rs = record_sets[0]
    main_rs_id = main_rs.id

    # Find all numeric fields
    numeric_fields = [
        f.id for f in getattr(main_rs, 'fields', [])
        if getattr(f, 'data_type', '').lower() in ('number', 'integer', 'float')
    ]
    print(f"Numeric fields in record set {main_rs_id}: {numeric_fields}")
    df = dataframes[main_rs_id]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        # Ensure numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        # Simple EDA: Remove nulls, filter, normalize
        initial_count = len(df)
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df.dropna(subset=[numeric_field])
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered {initial_count-len(filtered_df)} records with {numeric_field} <= mean ({threshold:.2f})")
        # Z-score normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Sample of {numeric_field} and its normalized values:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical field if available
        group_fields = [
            f.id for f in getattr(main_rs, 'fields', [])
            if getattr(f, 'data_type', '').lower() in ('string', 'text') and f.id != numeric_field
        ]
        group_field = group_fields[0] if group_fields else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(f"\nGrouped by '{group_field}', showing mean of {numeric_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in the main record set.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset with matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field if available
if 'filtered_df' in locals() and not filtered_df.empty and numeric_field in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group field available, show boxplot
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR-compliant scientific dataset using the `mlcroissant` library. We loaded the dataset metadata, explored record sets and fields (referenced by `@id`), extracted tabular data, performed simple EDA and grouping, and visualized key dataset attributes. For downstream analysis (machine learning or statistical tests), the structured approach above enables robust, reproducible workflows using variable and `@id` referencing.

**Next steps:** Try exploring additional record sets, merging DataFrames on shared keys, handling missing data, or building ML models using the prepared data.